In [68]:
import CC_Method_Analysis_v_0_1 as CC
import pandas as pd
from scipy import stats
import numpy as np
from docx import Document 
from docx.shared import Inches, Cm, Pt
import pandas as pd


from importlib import reload
reload(CC)

#______________Test and Error information____________________________

Test_Analyte = 'Lp(a)'
Test_Unit = 'nmol/L'
Input_file_name = "Liprotein_a.xlsx"
Output_file_name = 'LinearityReport.docx'

Error_level_cut_off = None # cut-off between absolute and percentage error
error1 = None # absolute error limit below the cut-off
error2 = 15 # percentage error limit above the cut-off
#___________________________________________________________

#open a blank document
document = Document()

#adjust page margins
sections = document.sections
for section in sections:
    section.top_margin = Cm(1.5)
    section.bottom_margin = Cm(1.5)
    section.left_margin = Cm(1.5)
    section.right_margin = Cm(1.5)

document.save(Output_file_name)

df_Linearity = pd.read_excel(Input_file_name, 
                      sheet_name='Linearity, Sr', 
                      usecols='A:F',
                      skiprows=range(0,14),
)

df_Linearity


Lvl_avg = df_Linearity.groupby('Levels')['Measured Result'].mean()
Lvl_sd = df_Linearity.groupby('Levels')['Measured Result'].std()
Lvl_exp = df_Linearity.groupby('Levels')['Expected Result'].mean()

df_Linearity_summary = pd.DataFrame({'Levels': Lvl_avg.index,
                                    'Measured Mean': Lvl_avg.values,
                                    'Measured SD': Lvl_sd.values,
                                    'Expected Mean': Lvl_exp.values
                                    })


# calculate the 95% confindence
# Use t-distribution for critical value
n = 2 
t_critical = stats.t.ppf((1 + 0.95) / 2, df=n-1)
t_critical

se = df_Linearity_summary['Measured SD']/np.sqrt(n)

df_Linearity_summary['Measured Mean Error'] =  t_critical * se #95% CI error margin to plot on the mean
df_Linearity_summary

CC.linearity_output(document = document,
                analyte= Test_Analyte,
                Measured_mean= df_Linearity_summary['Measured Mean'],
                Expected_mean= df_Linearity_summary['Expected Mean'], 
                Measured_err= df_Linearity_summary['Measured Mean Error'],
                n=n,
                Error_level_cut_off=Error_level_cut_off,
                error1=error1,
                error2=error2,
                Unit= Test_Unit
                )                
document.save(Output_file_name)


In [17]:
Lvl_avg

Levels
Level 1     322.35
Level 10     13.85
Level 2     272.80
Level 3     196.60
Level 4     169.50
Level 5     136.00
Level 6     115.60
Level 7      94.05
Level 8      71.30
Level 9      50.55
Name: Measured Result, dtype: float64

In [ ]:
CC.linearity_output(document = Document("MethodComparison.docx"),
                analyte= Test_Analyte,
                Measured_mean= df_Linearity_summary['Measured Mean'],
                Expected_mean= df_Linearity_summary['Expected Mean'], 
                Measured_err= df_Linearity_summary['Measured Mean Error'],
                n=3,
                Error_level_cut_off=Error_level_cut_off,
                error1=error1,
                error2=error2,
                Unit= Test_Unit
                )      